In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(message)s",
    force=True
)

logger = logging.getLogger("thermal_pipeline")

def load_raw_data(file_path):
    file_path = Path(file_path)

    if not file_path.is_file():
        logger.error("Raw data file not found: %s", file_path)
        raise FileNotFoundError(f"File not found: {file_path}")

    raw_df = pd.read_csv(
        file_path,
        dtype=str,
        keep_default_na=False,
        skip_blank_lines=False
    )

    required_columns = {"timestamp", "message"}
    missing_columns = required_columns - set(raw_df.columns)

    if missing_columns:
        logger.error("Missing required columns: %s", sorted(missing_columns))
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    logger.info(
        "Loaded %d rows from %s",
        len(raw_df),
        file_path.name
    )

    return raw_df

file_path = "data/raw/incubator_bt_log2.csv"

raw_df = load_raw_data(file_path)

clean_df = raw_df.copy()
clean_df = clean_df.rename(
    columns={
        "timestamp": "timestamp_raw",
        "message": "raw_message"
    }
)

clean_df.insert(0, "record_id", range(1, len(clean_df) + 1))

assert len(clean_df) == len(raw_df)
assert clean_df["timestamp_raw"].equals(raw_df["timestamp"])
assert clean_df["raw_message"].equals(raw_df["message"])

print("Raw rows:", len(raw_df))
print("Working rows:", len(clean_df))

print(
    clean_df[
        ["record_id", "timestamp_raw", "raw_message"]
    ].head(3).to_string(index=False)
)

INFO | Loaded 53075 rows from incubator_bt_log2.csv


Raw rows: 53075
Working rows: 53075
 record_id   timestamp_raw                                                                                                                                                                                                      raw_message
         1 6/24/2026 14:32 S1: 37.8C 60.0% P | S2: 37.9C 60.0% A | S3: 37.3C 60.0% A | S4: 38.6C 59.0% P | T=37.6 | H=60.0 | Duty=25 | Corr=0 | Htr=0 | HFan=0 | Servo=MAX | Mod=NRML | Sprd=0.6 | AvgSens=2 | RotMin=145 | LEC=0 | HEC=115
         2 6/24/2026 14:32 S1: 37.8C 60.0% P | S2: 37.9C 60.0% A | S3: 37.3C 60.0% A | S4: 38.6C 59.0% P | T=37.6 | H=60.0 | Duty=25 | Corr=0 | Htr=1 | HFan=0 | Servo=MAX | Mod=NRML | Sprd=0.6 | AvgSens=2 | RotMin=145 | LEC=0 | HEC=122
         3 6/24/2026 14:33 S1: 37.8C 60.0% P | S2: 37.9C 60.0% A | S3: 37.3C 60.0% A | S4: 38.6C 59.0% P | T=37.6 | H=60.0 | Duty=25 | Corr=0 | Htr=0 | HFan=0 | Servo=MAX | Mod=NRML | Sprd=0.6 | AvgSens=2 | RotMin=144 | LEC=0 | HEC=129


In [2]:
sample_message = clean_df["raw_message"].iloc[0]
message_parts = sample_message.split("|")

for part_number, part in enumerate(message_parts, start=1):
    print(part_number, part.strip())

1 S1: 37.8C 60.0% P
2 S2: 37.9C 60.0% A
3 S3: 37.3C 60.0% A
4 S4: 38.6C 59.0% P
5 T=37.6
6 H=60.0
7 Duty=25
8 Corr=0
9 Htr=0
10 HFan=0
11 Servo=MAX
12 Mod=NRML
13 Sprd=0.6
14 AvgSens=2
15 RotMin=145
16 LEC=0
17 HEC=115


In [3]:
empty_message = clean_df["raw_message"].str.strip().eq("")

section_count = clean_df["raw_message"].str.count(r"\|") + 1
section_count = section_count.mask(empty_message, 0)

section_summary = section_count.value_counts().sort_index()

if empty_message.sum() > 0:
    logger.warning(
        "Found %d empty raw messages",
        empty_message.sum()
    )

if len(section_summary) > 1:
    logger.warning(
        "Found messages with different numbers of sections"
    )

print(section_summary)

WARNING | Found messages with different numbers of sections


raw_message
2         1
3         4
4         3
14        1
16        5
17    53061
Name: count, dtype: int64


In [4]:
expected_sections = 17

message_text = clean_df["raw_message"].str.strip()

clean_df["message_is_empty"] = message_text.eq("")

clean_df["message_section_count"] = (
    message_text.str.count(r"\|") + 1
)

clean_df.loc[
    clean_df["message_is_empty"],
    "message_section_count"
] = 0

In [5]:
clean_df["message_status"] = "complete"

clean_df.loc[
    clean_df["message_is_empty"],
    "message_status"
] = "empty"

clean_df.loc[
    (~clean_df["message_is_empty"]) &
    (clean_df["message_section_count"] < expected_sections),
    "message_status"
] = "incomplete"

clean_df.loc[
    clean_df["message_section_count"] > expected_sections,
    "message_status"
] = "extra_sections"

In [6]:
message_summary = (
    clean_df["message_status"]
    .value_counts()
    .rename_axis("message_status")
    .reset_index(name="row_count")
)

message_summary["percentage"] = (
    message_summary["row_count"] / len(clean_df) * 100
).round(4)

print(message_summary)

  message_status  row_count  percentage
0       complete      53061     99.9736
1     incomplete         14      0.0264


In [7]:
problem_rows = clean_df[
    clean_df["message_status"] != "complete"
]

if not problem_rows.empty:
    logger.warning(
        "Found %d rows with incomplete, empty, or unexpected messages",
        len(problem_rows)
    )

print(
    problem_rows[
        [
            "record_id",
            "timestamp_raw",
            "message_section_count",
            "message_status",
            "raw_message"
        ]
    ].to_string(index=False)
)

WARNING | Found 14 rows with incomplete, empty, or unexpected messages


 record_id   timestamp_raw  message_section_count message_status                                                                                                                                                                                                          raw_message
      4544 6/25/2026 11:55                     16     incomplete      S1: 37.8C 55.0% P | S2: 38.0C 55.0% A l S3: 37.2C 56.0% A | S4: 39.1C 53.0% P | T=37.6 | H=55.5 | Duty=30 | Corr=5 | Htr=0 | HFan=0 | Servo=MAX | Mod=NRML | Sprd=0.8 | AvgSens=2 | RotMin=117 | LEC=0 | HEC=17
      9084  6/26/2026 8:40                     16     incomplete       S1: 37.9C 54.0% P | S2: 37.9C 53.0% A | S3: 37.1C 55.0% A | S4: 39.1C 50.0% P < T=37.5 | H=54.0 | Duty=40 | Corr=10 | Htr=1 | HFan=0 | Servo=MAX | Mod=NRML | Sprd=0.8 | AvgSens=2 | RotMin=79 | LEC=0 | HEC=0
     19431 6/28/2026 10:24                     14     incomplete                                                         S3: 37.2C 57.0% A | S4: 38.6C 57.6 | H=56.5 |